# Phase 6 — Dual-Path Stage 1 Training (ST-CDGM)

## Architecture
```
Path A (causal, frozen after Phase III starts):  DAG → GNN → RCN → head → μ_A
Path B (new, spatial, trainable):                LR grid → PathBCNN → μ_B
Gate (learnable):                                [μ_A, μ_B] → g ∈ (0,1)
Fusion:  μ_total = g·μ_B + (1-g)·μ_A
```

## Three-Phase Training Schedule
| Phase | Who trains       | Loss                                       | Epochs |
|-------|------------------|--------------------------------------------|--------|
| I     | Path B only      | MSE(μ_B, HR)                               | 7      |
| II    | Gate only        | MSE(μ_total, HR) + λ_div·L_div             | 3      |
| III   | Joint (A_dag ❄) | λ_c·MSE(μ_A,HR) + (1-λ_c)·MSE(μ_total,HR) + λ_div·L_div | 7 |

## Invariants
| Invariant            | Control                             |
|----------------------|-------------------------------------|
| A_dag frozen always  | requires_grad_(False) + drift<1e-5  |
| causal_frac ≥ 0.30   | diversity_loss + λ_c schedule       |
| RMSE(μ_total) < 0.20 | hard threshold Phase III            |
| std(μ_total) > std(μ_A) | causal signal preserved          |

## Inputs
- Path A checkpoint (Phase 5 best): `epoch_best_stage1_with_sre_best.pth`
- Non-causal init for Path B: `ckpt_noncausal/*.pth`

## Outputs
- `epoch_best_dualpath.pth` : encoder+RCN+head+DualPath
- `sigma_data_dualpath.json` : new σ_data for Stage 2

In [ ]:
# === Cell 1 : Bootstrap Colab ===
import subprocess, shlex, os, sys
from pathlib import Path

REPO_DIR   = Path('/content/climate_data')
GIT_URL    = 'https://github.com/leonelkenfack/stcdgm.git'
GIT_BRANCH = 'four-node-causal'

if not (REPO_DIR / '.git').exists():
    subprocess.run(shlex.split(
        f'git clone --depth 200 -b {GIT_BRANCH} {GIT_URL} {REPO_DIR}'), check=True)
else:
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} fetch --depth=200 origin {GIT_BRANCH}'), check=True)
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} reset --hard origin/{GIT_BRANCH}'), check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))

try:
    import torch_geometric; import cftime; import h5netcdf; import xbatcher; import diffusers
    from omegaconf import OmegaConf
except ImportError:
    EXTRA_DEPS = [
        'torch_geometric', 'omegaconf==2.3.0', 'hydra-core==1.3.2',
        'diffusers==0.36.0', 'einops', 'scipy', 'h5py', 'netCDF4',
        'xarray', 'dask', 'zarr', 'safetensors==0.7.0',
        'xbatcher', 'webdataset', 'cftime', 'h5netcdf',
    ]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + EXTRA_DEPS, check=True)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    print('[bootstrap] not on Colab — Drive mount skipped')

print(f'[bootstrap] cwd={os.getcwd()}  branch={GIT_BRANCH}')

In [ ]:
# === Cell 2 : Imports + Constantes ===
import json
import glob as _glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from contextlib import nullcontext
from pathlib import Path
from omegaconf import OmegaConf
from torch.optim import Adam
from torch.optim.lr_scheduler import LinearLR

from st_cdgm.models.dual_path_stage1 import DualPathPredictor, PathBCNN, FusionGate
from st_cdgm.training.stage1_paths import (
    train_epoch_dualpath_phase1,
    train_epoch_dualpath_phase2,
    train_epoch_dualpath_phase3,
    predict_mu_hr_dualpath,
    batch_lr_grid_last,
)

DRIVE_ROOT         = Path('/content/drive/MyDrive/climate_data')
ORACLE_9N          = DRIVE_ROOT / 'oracle_9node' / 'seed_42'
CKPT_PHASE5        = ORACLE_9N / 'epoch_best_stage1_with_sre_best.pth'
CKPT_NONCAUSAL_DIR = ORACLE_9N / 'ckpt_noncausal'
OUT_DIR            = ORACLE_9N

PHASE6_DIR         = OUT_DIR / 'phase6_dualpath_v3'
PHASE6_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PHASE_I_LAST  = PHASE6_DIR / 'phase_I_last.pth'
CKPT_PHASE_I_BEST  = PHASE6_DIR / 'phase_I_best.pth'
CKPT_PHASE_II_LAST = PHASE6_DIR / 'phase_II_last.pth'
CKPT_PHASE_II_BEST = PHASE6_DIR / 'phase_II_best.pth'
CKPT_PHASE_III_LAST= PHASE6_DIR / 'phase_III_last.pth'
CKPT_PHASE_III_BEST= PHASE6_DIR / 'phase_III_best.pth'
CKPT_DUAL_OUT      = OUT_DIR / 'epoch_best_dualpath.pth'
SIGMA_JSON         = OUT_DIR / 'sigma_data_dualpath.json'

RESUME = False

assert CKPT_PHASE5.exists(), f'Checkpoint Phase 5 introuvable : {CKPT_PHASE5}'
print(f'[Cell 2] Checkpoint Phase 5  : {CKPT_PHASE5}')
print(f'[Cell 2] Phase 6 dir         : {PHASE6_DIR}')
print(f'[Cell 2] Resume mode         : {RESUME}')

# --- Phase I : Path B standalone ---
PHASE_I_EPOCHS    = 5
PHASE_I_LR        = 2e-4
ADAM_BETA1        = 0.9
ADAM_BETA2        = 0.99
TAIL_WEIGHT_ALPHA = 5.0
WARMUP_STEPS      = 1000
SCALER_INIT_SCALE = 2 ** 13
SAFETY_STD_MIN    = 0.015
# v3 final crash : AMP fp16 sur UNet diffusers a produit des NaN persistants
# apres ~3000 batches (overflow attention/groupnorm). Pour stabilite max,
# Phase I tourne en fp32 (+~50% temps mais aucun NaN).
FORCE_FP32_PHASE_I = True

# --- Phase II : Gate only ---
PHASE_II_EPOCHS = 3
PHASE_II_LR     = 1e-3
LAMBDA_DIV      = 0.5

# --- Phase III : Joint fine-tuning (A_dag gele) ---
PHASE_III_EPOCHS        = 7
PHASE_III_LR_BACKBONE   = 1e-4
PHASE_III_LR_DUALPATH   = 2e-4
LAMBDA_CAUSAL_START     = 1.0
LAMBDA_CAUSAL_END       = 0.3
LAMBDA_DIV_III          = 0.5
GRADIENT_CLIPPING       = 1.0

PATH_B_KIND          = 'unet'
PATH_B_UNET_CHANNELS = (32, 64, 128)
PATH_B_UNET_LR_SHAPE = (23, 26)
PATH_B_BASE_CH       = 48
H_HR, W_HR           = 172, 179
GATE_MAX_MEAN        = 0.40

CRITERION_RMSE_IMPROVE = 0.95
CRITERION_RMSE_ABS     = 0.20
CRITERION_CAUSAL_FRAC  = 0.30
CRITERION_DAG_DRIFT    = 1e-5

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[Cell 2] DEVICE={DEVICE}')
print(f'[Cell 2] Phase I  : {PHASE_I_EPOCHS} ep  LR={PHASE_I_LR}  '
      f'alpha={TAIL_WEIGHT_ALPHA}  warmup={WARMUP_STEPS}  grad_clip={GRADIENT_CLIPPING}')
print(f'[Cell 2] AMP Phase I: {"FP32 force (stabilite max)" if FORCE_FP32_PHASE_I else "fp16 actif"}')
print(f'[Cell 2] PathB    : kind={PATH_B_KIND}  channels={PATH_B_UNET_CHANNELS}  (~3.6M params)')


In [ ]:
# === Cell 3 : Config + Pipeline + Dataloaders (9-node) ===
from torch.utils.data import DataLoader as _DataLoader, IterableDataset
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from path_c_plus.scripts.option_c_helpers import PATHCPLUS_HYPERPARAM_OVERRIDES
from path_c_plus.scripts.gpu_detect import detect_gpu_profile, print_profile_banner

# --- Config ---
CONFIG = OmegaConf.load('config/training_config.yaml')
_corrdiff = OmegaConf.load('config/training_config_corrdiff_normal.yaml')
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)

EXTENDED_9NODE = True
GPU_PROFILE = detect_gpu_profile()
print_profile_banner(GPU_PROFILE)

# Forcer batch_size=1 pour la compatibilité single-sample (IterableDataset)
CONFIG.training.batch_size  = 1
CONFIG.training.use_amp     = GPU_PROFILE['use_amp']
CONFIG.training.num_workers = GPU_PROFILE['num_workers']

ts_cfg = CONFIG.two_stage
ts_cfg.stage1['lambda_dag_prior'] = PATHCPLUS_HYPERPARAM_OVERRIDES['lambda_dag_prior']
ts_cfg.stage1['g_phys_alpha']     = PATHCPLUS_HYPERPARAM_OVERRIDES['g_phys_alpha']

# --- Ajout métapaths 9-node ---
OmegaConf.set_struct(CONFIG, False)
_existing_mp = {m.name for m in CONFIG.encoder.metapaths}
for _m in [
    {'name': 'Q850', 'src': 'Q850', 'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
    {'name': 'W500', 'src': 'W500', 'relation': 'causes', 'target': 'GP500', 'pool': 'mean'},
    {'name': 'IVT',  'src': 'IVT',  'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
]:
    if _m['name'] not in _existing_mp:
        CONFIG.encoder.metapaths.append(OmegaConf.create(_m))
print(f'[Cell 3] metapaths -> {[m.name for m in CONFIG.encoder.metapaths]}')

# --- Dates ---
K9_DATES = {
    'train':   ['1980-01-01', '2009-12-31'],
    'val':     ['2010-01-01', '2011-12-31'],
    'test':    ['2012-01-01', '2013-12-31'],
    'holdout': ['2014-01-01', '2014-12-31'],
}

_ON_COLAB  = 'google.colab' in sys.modules or Path('/content').exists()
DATA_ROOT  = Path('/content/drive/MyDrive/climate_data/data') if _ON_COLAB else Path('data/raw')
LR_PATH    = str(DATA_ROOT / 'train' / 'predictor_ACCESS-CM2_hist.nc')
HR_PATH    = str(DATA_ROOT / 'train' / 'pr_ACCESS-CM2_hist.nc')
_static_p  = DATA_ROOT / 'static_predictors' / 'ERA5_eval_ccam_12km.198110_NZ_Invariant.nc'
_mean_p    = DATA_ROOT / 'train' / 'means_ACCESS-CM2.nc'
_std_p     = DATA_ROOT / 'train' / 'stds_ACCESS-CM2.nc'
STATIC_PATH = str(_static_p) if _static_p.exists() else None
MEAN_PATH   = str(_mean_p)   if _mean_p.exists()   else None
STD_PATH    = str(_std_p)    if _std_p.exists()    else None

SEQ_LEN             = int(CONFIG.data.seq_len)
BASELINE_STRATEGY   = str(CONFIG.data.baseline_strategy)
BASELINE_FACTOR     = int(CONFIG.data.baseline_factor)
NORMALIZE           = bool(CONFIG.data.normalize)
PRECIPITATION_DELTA = float(CONFIG.data.precipitation_delta)
NAN_FILL_STRATEGY   = str(CONFIG.data.nan_fill_strategy)
_default_lr = ['q_500', 'q_850', 'u_500', 'u_850', 'v_500', 'v_850', 't_500', 't_850']
LR_VARIABLES  = list(CONFIG.data.lr_variables)  if CONFIG.data.get('lr_variables')  else _default_lr
HR_VARIABLES  = list(CONFIG.data.hr_variables)  if CONFIG.data.get('hr_variables')  else ['pr']
STATIC_VARIABLES = list(CONFIG.data.static_variables) if CONFIG.data.get('static_variables') else []

pipeline = NetCDFDataPipeline(
    lr_path=LR_PATH, hr_path=HR_PATH, static_path=STATIC_PATH,
    seq_len=SEQ_LEN, baseline_strategy=BASELINE_STRATEGY,
    baseline_factor=BASELINE_FACTOR, normalize=NORMALIZE,
    nan_fill_strategy=NAN_FILL_STRATEGY,
    precipitation_delta=PRECIPITATION_DELTA,
    lr_variables=LR_VARIABLES, hr_variables=HR_VARIABLES,
    static_variables=STATIC_VARIABLES,
    means_path=MEAN_PATH, stds_path=STD_PATH,
    train_start_date=K9_DATES['train'][0], train_end_date=K9_DATES['train'][1],
    val_start_date=K9_DATES['val'][0],     val_end_date=K9_DATES['val'][1],
    test_start_date=K9_DATES['test'][0],   test_end_date=K9_DATES['test'][1],
    temporal_holdout_start_date=K9_DATES['holdout'][0],
    temporal_holdout_end_date=K9_DATES['holdout'][1],
)
print('[Cell 3] Pipeline ready')

train_dataset = pipeline.build_sequence_dataset(split='train', seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)
val_dataset   = pipeline.build_sequence_dataset(split='val',   seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)

BATCH_SIZE  = 1
NUM_WORKERS = int(CONFIG.training.num_workers)
PIN_MEMORY  = bool(torch.cuda.is_available())
_loader_kw  = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                   pin_memory=PIN_MEMORY, collate_fn=lambda x: x)
if NUM_WORKERS > 0:
    _loader_kw.update(persistent_workers=True, prefetch_factor=2)

train_dataloader = _DataLoader(train_dataset,
                                shuffle=not isinstance(train_dataset, IterableDataset),
                                **_loader_kw)
val_dataloader   = _DataLoader(val_dataset, shuffle=False, **_loader_kw)

lr_shape = tuple(CONFIG.graph.lr_shape)
hr_shape = tuple(CONFIG.graph.hr_shape)
builder  = HeteroGraphBuilder(
    lr_shape=lr_shape, hr_shape=hr_shape,
    static_dataset=pipeline.get_static_dataset(),
    include_mid_layer=CONFIG.graph.include_mid_layer,
    extended_9node=EXTENDED_9NODE,
)
print(f'[Cell 3] Builder  lr_shape={lr_shape}  hr_shape={hr_shape}  dyn={builder.dynamic_node_types}')

# --- convert_sample_to_batch (identique phase5, avec lr_grid pour batch_lr_grid_last) ---
_LR_VARS = list(CONFIG.data.lr_variables)
_VI = {v: i for i, v in enumerate(_LR_VARS)}
_Q_IDX = [_VI[v] for v in ('q_850', 'q_500', 'q_250') if v in _VI]
_W_IDX = [_VI[v] for v in ('w_850', 'w_500', 'w_250') if v in _VI]
_IVT_LEVELS = [lev for lev in ('850', '500', '250')
               if f'q_{lev}' in _VI and f'u_{lev}' in _VI and f'v_{lev}' in _VI]

def _compute_ivt_nodes(lr0):
    acc = None
    for lev in _IVT_LEVELS:
        q = lr0[:, [_VI[f'q_{lev}']]]
        u = lr0[:, [_VI[f'u_{lev}']]]
        v = lr0[:, [_VI[f'v_{lev}']]]
        term = q * torch.sqrt(u * u + v * v + 1e-12)
        acc = term if acc is None else acc + term
    if acc is None:
        acc = lr0[:, 0:1] * 0.0
    return acc / (len(_IVT_LEVELS) + 1e-8)

def _ensure_2d(t):
    return t.unsqueeze(-1) if t.dim() == 1 else t

def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample['lr']
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    lr0 = lr_nodes_steps[0]
    if EXTENDED_9NODE:
        _ivt = _compute_ivt_nodes(lr0)
        dynamic_features = {}
        for nt in builder.dynamic_node_types:
            if nt == 'Q850':   dynamic_features[nt] = _ensure_2d(lr0[:, _Q_IDX] if _Q_IDX else lr0)
            elif nt == 'W500': dynamic_features[nt] = _ensure_2d(lr0[:, _W_IDX] if _W_IDX else lr0)
            elif nt == 'IVT':  dynamic_features[nt] = _ensure_2d(_ivt)
            else:              dynamic_features[nt] = _ensure_2d(lr0)
    else:
        dynamic_features = {nt: _ensure_2d(lr0) for nt in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)
    return {
        'lr':       lr_tensor,
        'lr_grid':  lr_seq,
        'residual': sample['residual'],
        'baseline': sample.get('baseline'),
        'hetero':   hetero,
        'time':     sample.get('time'),
    }

def iterate_batches(dataloader, builder, device):
    for batch_list in dataloader:
        if not isinstance(batch_list, list):
            batch_list = [batch_list]
        yield [convert_sample_to_batch(s, builder, device) for s in batch_list]

_probe = next(iter(train_dataset))
C_LR = _probe['lr'].shape[1]
_n_train = len(train_dataset) if hasattr(train_dataset, '__len__') else '?'
_n_val   = len(val_dataset)   if hasattr(val_dataset,   '__len__') else '?'
print(f'[Cell 3] C_LR={C_LR}  lr_shape={lr_shape}  hr_shape={hr_shape}')
print(f'[Cell 3] train={_n_train} samples  val={_n_val} samples')

_s = convert_sample_to_batch(next(iter(train_dataset)), builder, 'cpu')
for nt in builder.dynamic_node_types:
    _f = _s['hetero'].x_dict.get(nt) if hasattr(_s['hetero'], 'x_dict') else None
    if _f is not None:
        assert _f.dim() == 2, f'WARN features {nt} sont {_f.dim()}D (attendu 2D)'
print('[Cell 3] Verification features 2D : OK')
USE_AMP = bool(CONFIG.training.use_amp)

In [ ]:
# === Cell 4 : Charger Path A depuis Phase 5 ===
import re
from st_cdgm.models.intelligible_encoder import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig,
)
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.regression_head import GraphToGridDecoder

def _parse_encoder_metapaths_from_ckpt(enc_sd):
    """Infer ordered (name, src, rel, target) list from checkpoint encoder keys.
    Format: metapath_convs.{name}__{src}__{rel}__{target}.{param}
    """
    seen = {}
    order = []
    for k in enc_sd:
        if not k.startswith('metapath_convs.'):
            continue
        rest = k[len('metapath_convs.'):]
        parts = rest.split('__')
        if len(parts) < 4:
            continue
        name = parts[0]
        src  = parts[1]
        rel  = parts[2]
        tgt  = parts[3].split('.')[0]
        if name not in seen:
            seen[name] = (src, rel, tgt)
            order.append(name)
    return [(n,) + seen[n] for n in order]

def _build_encoder_from_ckpt(enc_sd, CONFIG, device):
    """Build encoder whose metapath configs match exactly the checkpoint keys."""
    parsed = _parse_encoder_metapaths_from_ckpt(enc_sd)
    print(f'  Metapaths detectes : {[t[0] for t in parsed]}')
    cfgs = [
        IntelligibleVariableConfig(name=name, meta_path=(src, rel, tgt), pool='mean')
        for name, src, rel, tgt in parsed
    ]
    enc = IntelligibleVariableEncoder(
        configs=cfgs,
        hidden_dim=int(CONFIG.encoder.hidden_dim),
        conditioning_dim=int(CONFIG.encoder.conditioning_dim),
    ).to(device)
    return enc, len(cfgs)

# --- Charger checkpoint Phase 5 ---
print(f'[Cell 4] Chargement : {CKPT_PHASE5}')
ck5 = torch.load(CKPT_PHASE5, map_location=DEVICE, weights_only=False)
print(f'[Cell 4] Cles disponibles dans le checkpoint : {list(ck5.keys())}')

# Normaliser les cles (supprimer prefixe _orig_mod si present)
def _clean_sd(sd):
    if sd is None:
        return None
    if any('_orig_mod' in k for k in sd):
        sd = {k.replace('_orig_mod.', ''): v for k, v in sd.items()}
    return sd

_enc_sd_raw = _clean_sd(ck5.get('encoder_state_dict', {}))

torch.manual_seed(SEED); np.random.seed(SEED)
print('[Cell 4] Inference structure encoder depuis checkpoint...')
encoder, num_vars = _build_encoder_from_ckpt(_enc_sd_raw, CONFIG, DEVICE)

# RCN driver dimension
_probe_b = next(iter(train_dataset))
_lr_nodes = builder.lr_grid_to_nodes(_probe_b['lr'][0])
RCN_DRIVER_DIM = _lr_nodes.shape[-1]

rcn_cell = RCNCell(
    num_vars=num_vars,
    hidden_dim=int(CONFIG.rcn.hidden_dim),
    driver_dim=RCN_DRIVER_DIM,
    reconstruction_dim=RCN_DRIVER_DIM,
    dropout=float(CONFIG.rcn.dropout),
).to(DEVICE)
rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get('detach_interval'))

rh_cfg = CONFIG.two_stage.regression_head
regression_head = GraphToGridDecoder(
    d_model=int(rh_cfg.d_model),
    hr_h=H_HR, hr_w=W_HR,
    intermediate_h=int(rh_cfg.intermediate_h),
    intermediate_w=int(rh_cfg.intermediate_w),
    n_heads=int(rh_cfg.n_heads),
    refine_channels=int(rh_cfg.refine_channels),
    output_channels=1,
).to(DEVICE)

# --- Charger les poids avec fallback cles alternatives ---
def _load_sd(module, keys_priority, ck_data, label):
    """Try keys in order until one is found in ck_data."""
    for key in keys_priority:
        sd = ck_data.get(key)
        if sd is not None:
            sd = _clean_sd(sd)
            module.load_state_dict(sd, strict=True)
            print(f'  {label} : charge depuis cle "{key}"')
            return
    print(f'  WARN {label} : aucune cle trouvee dans {keys_priority}')

_load_sd(encoder,         ['encoder_state_dict'],                                  ck5, 'encoder')
_load_sd(rcn_cell,        ['rcn_cell_state_dict', 'rcn_state_dict'],               ck5, 'rcn_cell')
_load_sd(regression_head, ['regression_head_state_dict', 'head_state_dict'],       ck5, 'regression_head')
# SRE n'est pas utilise en Phase 6 — on l'ignore volontairement

_n_enc = sum(p.numel() for p in encoder.parameters())
_n_rcn = sum(p.numel() for p in rcn_cell.parameters())
_n_rh  = sum(p.numel() for p in regression_head.parameters())
print(f'[Cell 4] Stack Path A charge  epoch={ck5.get("epoch", ck5.get("phase_b_epoch", "?"))}  '
      f'enc={_n_enc:,}  rcn={_n_rcn:,}  head={_n_rh:,}')

# --- Geler A_dag ---
_rcn_core = rcn_cell
if hasattr(_rcn_core, '_orig_mod'):
    _rcn_core = _rcn_core._orig_mod
if hasattr(_rcn_core, 'A_dag'):
    _rcn_core.A_dag.requires_grad_(False)
    _A_dag_ref = _rcn_core.A_dag.detach().clone()
    print(f'[Cell 4] A_dag gele  shape={tuple(_rcn_core.A_dag.shape)}  '
          f'norm={_A_dag_ref.norm():.4f}  '
          f'asym={(_A_dag_ref - _A_dag_ref.T).abs().mean():.4f}')
else:
    _A_dag_ref = None
    print('[Cell 4] WARN : A_dag introuvable dans rcn_cell')

def _freeze_path_a():
    for p in encoder.parameters():         p.requires_grad_(False)
    for p in rcn_cell.parameters():        p.requires_grad_(False)
    for p in regression_head.parameters(): p.requires_grad_(False)
    if _A_dag_ref is not None:
        _rcn_core.A_dag.requires_grad_(False)
    print('[freeze] Path A (encoder+RCN+head) gele.')

def _unfreeze_path_a_keep_dag():
    for p in encoder.parameters():         p.requires_grad_(True)
    for p in rcn_cell.parameters():        p.requires_grad_(True)
    for p in regression_head.parameters(): p.requires_grad_(True)
    if _A_dag_ref is not None:
        _rcn_core.A_dag.requires_grad_(False)  # A_dag reste gele
    print('[unfreeze] Path A degele. A_dag reste gele.')

def _check_dag_drift(label=''):
    if _A_dag_ref is None:
        return 0.0
    drift = (_rcn_core.A_dag.detach() - _A_dag_ref).abs().max().item()
    if label:
        print(f'  A_dag drift={drift:.1e}  [{label}]')
    return drift

_freeze_path_a()

# --- Probe baseline RMSE(mu_A) sur quelques samples val ---
@torch.no_grad()
def _probe_mu_a(n_samples=30):
    encoder.eval(); rcn_cell.eval(); regression_head.eval()
    errs = []
    for i, batch_list in enumerate(iterate_batches(val_dataloader, builder, DEVICE)):
        if i >= n_samples:
            break
        for micro in batch_list:
            tgt = micro['residual'][-1].to(DEVICE)
            if tgt.dim() == 3: tgt = tgt.unsqueeze(0)
            valid = torch.isfinite(tgt)
            lr_data = micro['lr'].to(DEVICE)
            h_init  = encoder.init_state(micro['hetero']).to(DEVICE)
            drivers = [lr_data[t] for t in range(lr_data.shape[0])]
            seq_out = rcn_runner.run(h_init, drivers, reconstruction_sources=None)
            mu_A = regression_head(seq_out.states[-1])
            if mu_A.dim() == 3: mu_A = mu_A.unsqueeze(0)
            if mu_A.shape != tgt.shape:
                mu_A = F.interpolate(mu_A, size=tgt.shape[-2:], mode='bilinear', align_corners=False)
            errs.append((mu_A[valid] - tgt[valid]).pow(2).mean().item())
    rmse_A = float(np.mean(errs)) ** 0.5
    print(f'[Cell 4] RMSE(mu_A) baseline sur {len(errs)} samples val : {rmse_A:.5f}')
    return rmse_A

RMSE_A_BASELINE = _probe_mu_a()
print(f'[Cell 4] RMSE_A_BASELINE = {RMSE_A_BASELINE:.5f}  (cible: < {RMSE_A_BASELINE * CRITERION_RMSE_IMPROVE:.5f})')

In [ ]:
# === Cell 5 : Initialiser DualPathPredictor ===

dual_path = DualPathPredictor(
    in_channels=C_LR,
    base_ch=PATH_B_BASE_CH,
    hr_h=H_HR,
    hr_w=W_HR,
    gate_max_mean=GATE_MAX_MEAN,
    path_b_kind=PATH_B_KIND,
    path_b_unet_channels=PATH_B_UNET_CHANNELS,
    path_b_unet_lr_shape=PATH_B_UNET_LR_SHAPE,
).to(DEVICE)

_n_path_b = sum(p.numel() for p in dual_path.path_b.parameters())
_n_gate   = sum(p.numel() for p in dual_path.gate.parameters())
print(f'DualPathPredictor: {dual_path.num_params():,} params')
print(f'  Path B ({PATH_B_KIND}): {_n_path_b:,} params')
print(f'  Gate              : {_n_gate:,} params')
print(f'  Gate init: g approx {torch.sigmoid(torch.tensor(-2.0)):.3f} -> mu_total approx mu_A')

# --- Essayer de charger poids non-causaux dans Path B ---
_noncausal_loaded = False
if CKPT_NONCAUSAL_DIR.exists():
    _pth_files = list(CKPT_NONCAUSAL_DIR.glob('*.pth'))
    if _pth_files:
        _nc_ckpt_path = max(_pth_files, key=lambda p: p.stat().st_mtime)
        print(f'[Cell 5] Checkpoint non-causal trouve : {_nc_ckpt_path}')
        try:
            _nc_ck = torch.load(_nc_ckpt_path, map_location=DEVICE, weights_only=False)
            if isinstance(_nc_ck, dict):
                _nc_sd = (_nc_ck.get('model_state_dict')
                          or _nc_ck.get('regression_head_state_dict')
                          or _nc_ck.get('state_dict')
                          or _nc_ck)
            else:
                _nc_sd = _nc_ck
            if isinstance(_nc_sd, dict):
                _nc_sd = {k.replace('_orig_mod.', ''): v for k, v in _nc_sd.items()}
                # En mode 'unet', les cles non-causales sont directement compatibles
                # (RegressionMeanPredictor) — pas besoin de remap path_b.*
                if PATH_B_KIND == 'unet':
                    _info = dual_path.path_b.load_state_dict(_nc_sd, strict=False)
                    if len(_info.missing_keys) < len(dict(dual_path.path_b.named_parameters())) * 0.5:
                        print(f'[Cell 5] Poids non-causaux charges (unet)  '
                              f'missing={len(_info.missing_keys)} unexpected={len(_info.unexpected_keys)}')
                        _noncausal_loaded = True
                else:
                    # mode 'cnn' : remap unet.* / hr_proj.* -> path_b.*
                    _mapped = {}
                    for k, v in _nc_sd.items():
                        if k.startswith('unet.') or k.startswith('hr_proj.'):
                            _mapped['path_b.' + k] = v
                        elif k.startswith('path_b.'):
                            _mapped[k] = v
                    if _mapped:
                        _info = dual_path.load_state_dict(_mapped, strict=False)
                        print(f'[Cell 5] Poids non-causaux charges (cnn remap)  '
                              f'missing={len(_info.missing_keys)} unexpected={len(_info.unexpected_keys)}')
                        _noncausal_loaded = True
        except Exception as e:
            print(f'[Cell 5] WARN : Echec chargement non-causal ({e}) — init aleatoire')
    else:
        print(f'[Cell 5] Aucun .pth dans {CKPT_NONCAUSAL_DIR} — init aleatoire Path B')
else:
    print(f'[Cell 5] Dossier non-causal absent ({CKPT_NONCAUSAL_DIR}) — init aleatoire Path B')

if not _noncausal_loaded:
    print('[Cell 5] WARNING : Path B initialisee aleatoirement (pas de warm-start non-causal).')
    print('  -> Le UNet 4.8M proven devrait converger en 3-5 epochs malgre tout.')

# Verifier zero-init gate
with torch.no_grad():
    _g_init = torch.sigmoid(dual_path.gate.conv_out.bias.data[0]).item()
print(f'[Cell 5] Gate bias init -> g_init ~ {_g_init:.3f}  (cible: < 0.15 -> mu_total ~ mu_A)')


In [ ]:
# === Cell 6 : Phase I — Path B standalone (CorrDiff-aligned + warmup + scaler) ===
print('=' * 60)
print('PHASE I : Path B seul (Path A et Gate geles)')
print(f'  {PHASE_I_EPOCHS} epochs  LR={PHASE_I_LR}  betas=({ADAM_BETA1},{ADAM_BETA2})')
print(f'  loss = plain MSE  (CorrDiff Mardani 2024)')
print(f'  warmup linear {WARMUP_STEPS} steps  scaler init_scale={SCALER_INIT_SCALE}')
print(f'  grad_clip={GRADIENT_CLIPPING}  safety std(mu_B) >= {SAFETY_STD_MIN} a ep2')
print('=' * 60)

# --- Setup gels ---
_freeze_path_a()
for p in dual_path.gate.parameters():
    p.requires_grad_(False)
for p in dual_path.path_b.parameters():
    p.requires_grad_(True)

# Adam avec betas CorrDiff
optimizer_I = Adam(
    dual_path.path_b.parameters(),
    lr=PHASE_I_LR,
    betas=(ADAM_BETA1, ADAM_BETA2),
    eps=1e-8,
)

# Warmup scheduler : LinearLR de 1% -> 100% sur WARMUP_STEPS
# Apres warmup, LR reste constante a PHASE_I_LR (pas de decay pour Phase I courte)
scheduler_I = LinearLR(
    optimizer_I,
    start_factor=0.01,
    end_factor=1.0,
    total_iters=WARMUP_STEPS,
)

# GradScaler persistant entre epochs avec init_scale serree
_use_amp_phase_I = (USE_AMP and DEVICE.type == 'cuda' and not FORCE_FP32_PHASE_I)
scaler_I = torch.amp.GradScaler(
    enabled=_use_amp_phase_I,
    init_scale=SCALER_INIT_SCALE,
)
print(f'  [Phase I] AMP enabled = {_use_amp_phase_I}')

# --- Resume detection ---
_start_ep    = 1
best_rmse_B  = float('inf')
phase_I_losses = []

if RESUME and CKPT_PHASE_I_LAST.exists():
    print(f'[Cell 6] RESUME : checkpoint trouve {CKPT_PHASE_I_LAST}')
    _ck_I = torch.load(CKPT_PHASE_I_LAST, map_location=DEVICE, weights_only=False)
    dual_path.path_b.load_state_dict(_ck_I['path_b_state_dict'])
    optimizer_I.load_state_dict(_ck_I['optimizer_state_dict'])
    if 'scheduler_state_dict' in _ck_I:
        scheduler_I.load_state_dict(_ck_I['scheduler_state_dict'])
    if 'scaler_state_dict' in _ck_I and scaler_I.is_enabled():
        scaler_I.load_state_dict(_ck_I['scaler_state_dict'])
    _start_ep      = int(_ck_I.get('epoch', 0)) + 1
    best_rmse_B    = float(_ck_I.get('best_rmse_B', float('inf')))
    phase_I_losses = list(_ck_I.get('phase_I_losses', []))
    print(f'  Reprise a epoch {_start_ep}/{PHASE_I_EPOCHS}  '
          f'best_rmse_B={best_rmse_B:.5f}')

    if _start_ep > PHASE_I_EPOCHS:
        print(f'[Cell 6] Phase I deja terminee.')
        if CKPT_PHASE_I_BEST.exists():
            _ck_best = torch.load(CKPT_PHASE_I_BEST, map_location=DEVICE, weights_only=False)
            dual_path.path_b.load_state_dict(_ck_best['path_b_state_dict'])
            best_rmse_B = float(_ck_best.get('best_rmse_B', best_rmse_B))

# --- Training loop ---
for ep in range(_start_ep, PHASE_I_EPOCHS + 1):
    print(f'\n--- Phase I epoch {ep}/{PHASE_I_EPOCHS} ---')
    m_I = train_epoch_dualpath_phase1(
        dual_path=dual_path,
        optimizer=optimizer_I,
        data_loader=iterate_batches(train_dataloader, builder, DEVICE),
        device=DEVICE,
        builder=builder,
        gradient_clipping=GRADIENT_CLIPPING,
        log_interval=30,
        use_amp=_use_amp_phase_I,
        verbose=True,
        tail_weight_alpha=TAIL_WEIGHT_ALPHA,
        lr_scheduler=scheduler_I,
        scaler=scaler_I,
    )
    _n_skip = m_I.get('n_skipped_nan', 0)
    _scale  = m_I.get('final_scale', 1.0)
    _cur_lr = optimizer_I.param_groups[0]['lr']
    print(f'  loss_B={m_I["loss_B"]:.5f}  n_batches={m_I["n_batches"]}  '
          f'n_skip_nan={_n_skip}  scale={_scale:.0f}  lr={_cur_lr:.2e}')
    if _n_skip > 0:
        _ratio = _n_skip / max(m_I['n_batches'] + _n_skip, 1)
        if _ratio > 0.05:
            print(f'  WARN : {_ratio*100:.1f}% batches skipped pour NaN — '
                  f'baisser LR ou activer fp32')
    phase_I_losses.append(m_I['loss_B'])

    # --- Probe val RMSE(mu_B) + std(mu_B) ---
    dual_path.path_b.eval()
    _errs_B = []
    _std_B_list = []
    with torch.no_grad():
        for _bi, _bl in enumerate(iterate_batches(val_dataloader, builder, DEVICE)):
            if _bi >= 50: break
            for _micro in _bl:
                _tgt = _micro['residual'][-1].to(DEVICE)
                if _tgt.dim() == 3: _tgt = _tgt.unsqueeze(0)
                _valid = torch.isfinite(_tgt)
                if not _valid.any(): continue
                _lr_g = batch_lr_grid_last(_micro, builder=builder, device=DEVICE)
                _lr_s = torch.nan_to_num(_lr_g, nan=0.0)
                _mu_B = dual_path.path_b(_lr_s)
                if _mu_B.shape != _tgt.shape:
                    _mu_B = F.interpolate(_mu_B, size=_tgt.shape[-2:], mode='bilinear', align_corners=False)
                _errs_B.append((_mu_B[_valid] - _tgt[_valid]).pow(2).mean().item())
                _std_B_list.append(_mu_B[_valid].std().item())
    _rmse_B_ep = float(np.mean(_errs_B)) ** 0.5 if _errs_B else float('inf')
    _std_B_ep  = float(np.mean(_std_B_list)) if _std_B_list else 0.0
    print(f'  RMSE(mu_B) val = {_rmse_B_ep:.5f}  std(mu_B) = {_std_B_ep:.4f}')
    dual_path.path_b.train()

    # --- Persistance : last + best ---
    _ck_payload = {
        'phase':                'I',
        'epoch':                ep,
        'path_b_state_dict':    dual_path.path_b.state_dict(),
        'optimizer_state_dict': optimizer_I.state_dict(),
        'scheduler_state_dict': scheduler_I.state_dict(),
        'scaler_state_dict':    scaler_I.state_dict() if scaler_I.is_enabled() else None,
        'best_rmse_B':          best_rmse_B,
        'rmse_B_epoch':         _rmse_B_ep,
        'std_B_epoch':          _std_B_ep,
        'phase_I_losses':       phase_I_losses,
        'tail_weight_alpha':    TAIL_WEIGHT_ALPHA,
        'path_b_kind':          PATH_B_KIND,
        'path_b_unet_channels': list(PATH_B_UNET_CHANNELS),
    }
    torch.save(_ck_payload, CKPT_PHASE_I_LAST)
    print(f'  Saved  -> {CKPT_PHASE_I_LAST.name}')

    if _rmse_B_ep < best_rmse_B and np.isfinite(_rmse_B_ep):
        best_rmse_B = _rmse_B_ep
        _ck_payload['best_rmse_B'] = best_rmse_B
        torch.save(_ck_payload, CKPT_PHASE_I_BEST)
        print(f'  * NEW best RMSE_B={best_rmse_B:.5f}  -> {CKPT_PHASE_I_BEST.name}')

    # --- Sanity A_dag ---
    _d = _check_dag_drift(f'Phase-I-ep{ep}')
    assert _d < CRITERION_DAG_DRIFT, f'A_dag a bouge ! drift={_d:.2e}'

    # --- Safety check epoch 2 : detecte effondrement ---
    if ep == 2 and _std_B_ep < SAFETY_STD_MIN:
        msg = (f'\nSAFETY ABORT : std(mu_B)={_std_B_ep:.4f} < {SAFETY_STD_MIN} '
               f'apres 2 epochs.\n'
               f'  Path B s\'effondre encore. Investiguer pipeline ou loss.')
        print(msg)
        raise RuntimeError(msg)

print(f'\n[Cell 6] Phase I terminee. best RMSE_B={best_rmse_B:.5f}')

# Charger le best pour la suite (Phase II)
if CKPT_PHASE_I_BEST.exists():
    _ck_best = torch.load(CKPT_PHASE_I_BEST, map_location=DEVICE, weights_only=False)
    dual_path.path_b.load_state_dict(_ck_best['path_b_state_dict'])
    print(f'  -> Path B charge depuis BEST (RMSE_B={_ck_best["best_rmse_B"]:.5f})')


In [ ]:
# === Cell 7 : Probe complet apres Phase I ===
print('=== Probe apres Phase I ===')

@torch.no_grad()
def _probe_dualpath(label='', n_max=None):
    """Probe val : RMSE(mu_A), RMSE(mu_B), RMSE(mu_total), gate_mean, causal_frac."""
    encoder.eval(); rcn_cell.eval(); regression_head.eval(); dual_path.eval()
    errs_A, errs_B, errs_total = [], [], []
    std_A_list, std_B_list, std_total_list = [], [], []
    gate_means, causal_fracs = [], []

    for bi, batch_list in enumerate(iterate_batches(val_dataloader, builder, DEVICE)):
        if n_max is not None and bi >= n_max:
            break
        for micro in batch_list:
            tgt = micro['residual'][-1].to(DEVICE)
            if tgt.dim() == 3: tgt = tgt.unsqueeze(0)
            valid = torch.isfinite(tgt)
            if not valid.any(): continue

            mu_A, mu_B, mu_total, gate = predict_mu_hr_dualpath(
                micro,
                encoder=encoder, rcn_runner=rcn_runner,
                regression_head=regression_head, dual_path=dual_path,
                builder=builder, device=DEVICE,
                target_shape=tgt.shape[-2:],
            )

            errs_A.append((mu_A[valid] - tgt[valid]).pow(2).mean().item())
            errs_B.append((mu_B[valid] - tgt[valid]).pow(2).mean().item())
            errs_total.append((mu_total[valid] - tgt[valid]).pow(2).mean().item())

            std_A_list.append(mu_A[valid].std().item())
            std_B_list.append(mu_B[valid].std().item())
            std_total_list.append(mu_total[valid].std().item())

            gate_means.append(gate.mean().item())
            causal_fracs.append(dual_path.causal_frac(mu_A, mu_B))

    def _m(lst): return float(np.mean(lst)) if lst else float('nan')

    rmse_A     = _m(errs_A)     ** 0.5
    rmse_B     = _m(errs_B)     ** 0.5
    rmse_total = _m(errs_total) ** 0.5
    std_A      = _m(std_A_list)
    std_B      = _m(std_B_list)
    std_total  = _m(std_total_list)
    gate_mean  = _m(gate_means)
    causal_frac = _m(causal_fracs)

    print(f'[probe {label}]')
    print(f'  RMSE : mu_A={rmse_A:.5f}  mu_B={rmse_B:.5f}  mu_total={rmse_total:.5f}')
    print(f'  std  : mu_A={std_A:.4f}   mu_B={std_B:.4f}   mu_total={std_total:.4f}')
    print(f'  gate_mean={gate_mean:.4f}  causal_frac={causal_frac:.3f}')
    return dict(
        rmse_A=rmse_A, rmse_B=rmse_B, rmse_total=rmse_total,
        std_A=std_A, std_B=std_B, std_total=std_total,
        gate_mean=gate_mean, causal_frac=causal_frac,
    )

probe_after_I = _probe_dualpath('APRES-PHASE-I')

print('\nTableau comparatif baseline vs apres Phase I :')
print(f'{"Metrique":<20} {"Baseline mu_A":>16} {"Phase I mu_B":>16}')
print('-' * 54)
print(f'{"RMSE":<20} {RMSE_A_BASELINE:>16.5f} {probe_after_I["rmse_B"]:>16.5f}')
print(f'{"std":<20} {"?":>16} {probe_after_I["std_B"]:>16.4f}')
print(f'{"causal_frac":<20} {1.0:>16.3f} {probe_after_I["causal_frac"]:>16.3f}')

In [ ]:
# === Cell 8 : Phase II  Gate uniquement (A et B geles, avec resume) ===
print('=' * 60)
print('PHASE II : Gate uniquement (Path A et Path B geles)')
print(f'  {PHASE_II_EPOCHS} epochs  LR={PHASE_II_LR}  lambda_div={LAMBDA_DIV}')
print('=' * 60)

# --- Setup gels ---
for p in dual_path.path_b.parameters():
    p.requires_grad_(False)
_freeze_path_a()
for p in dual_path.gate.parameters():
    p.requires_grad_(True)

optimizer_II = Adam(dual_path.gate.parameters(), lr=PHASE_II_LR)

# --- Resume detection ---
_start_ep_II   = 1
best_rmse_II   = float('inf')
phase_II_metrics = []

if RESUME and CKPT_PHASE_II_LAST.exists():
    print(f'[Cell 8] RESUME : checkpoint trouve {CKPT_PHASE_II_LAST}')
    _ck_II = torch.load(CKPT_PHASE_II_LAST, map_location=DEVICE, weights_only=False)
    dual_path.path_b.load_state_dict(_ck_II['path_b_state_dict'])
    dual_path.gate.load_state_dict(_ck_II['gate_state_dict'])
    optimizer_II.load_state_dict(_ck_II['optimizer_state_dict'])
    _start_ep_II     = int(_ck_II.get('epoch', 0)) + 1
    best_rmse_II     = float(_ck_II.get('best_rmse_total', float('inf')))
    phase_II_metrics = list(_ck_II.get('phase_II_metrics', []))
    print(f'  Reprise a epoch {_start_ep_II}/{PHASE_II_EPOCHS}  '
          f'best_rmse_total={best_rmse_II:.5f}')

    if _start_ep_II > PHASE_II_EPOCHS:
        print(f'[Cell 8] Phase II deja terminee.')
        if CKPT_PHASE_II_BEST.exists():
            _ck_best = torch.load(CKPT_PHASE_II_BEST, map_location=DEVICE, weights_only=False)
            dual_path.gate.load_state_dict(_ck_best['gate_state_dict'])
            best_rmse_II = float(_ck_best.get('best_rmse_total', best_rmse_II))
            print(f'  Gate charge depuis BEST (RMSE_total={best_rmse_II:.5f})')

# --- Training loop ---
for ep in range(_start_ep_II, PHASE_II_EPOCHS + 1):
    print(f'\n--- Phase II epoch {ep}/{PHASE_II_EPOCHS} ---')
    m_II = train_epoch_dualpath_phase2(
        dual_path=dual_path,
        optimizer=optimizer_II,
        data_loader=iterate_batches(train_dataloader, builder, DEVICE),
        device=DEVICE,
        encoder=encoder,
        rcn_runner=rcn_runner,
        regression_head=regression_head,
        builder=builder,
        lambda_div=LAMBDA_DIV,
        gradient_clipping=GRADIENT_CLIPPING,
        log_interval=30,
        use_amp=USE_AMP,
        verbose=True,
    )
    print(f'  loss_total={m_II["loss_total"]:.5f}  loss_div={m_II["loss_div"]:.5f}')

    # --- Probe partielle (50 batches val) ---
    _p = _probe_dualpath(f'Phase-II-ep{ep}', n_max=50)
    print(f'  gate_mean={_p["gate_mean"]:.4f}  causal_frac={_p["causal_frac"]:.3f}  '
          f'rmse_total={_p["rmse_total"]:.5f}')
    phase_II_metrics.append({**m_II, **_p})

    # --- Persistance : last + best ---
    _ck_payload_II = {
        'phase':                'II',
        'epoch':                ep,
        'path_b_state_dict':    dual_path.path_b.state_dict(),
        'gate_state_dict':      dual_path.gate.state_dict(),
        'optimizer_state_dict': optimizer_II.state_dict(),
        'best_rmse_total':      best_rmse_II,
        'rmse_total_epoch':     _p['rmse_total'],
        'gate_mean_epoch':      _p['gate_mean'],
        'causal_frac_epoch':    _p['causal_frac'],
        'phase_II_metrics':     phase_II_metrics,
    }
    torch.save(_ck_payload_II, CKPT_PHASE_II_LAST)
    print(f'  Saved  -> {CKPT_PHASE_II_LAST.name}')

    if _p['rmse_total'] < best_rmse_II:
        best_rmse_II = _p['rmse_total']
        _ck_payload_II['best_rmse_total'] = best_rmse_II
        torch.save(_ck_payload_II, CKPT_PHASE_II_BEST)
        print(f'  * NEW best RMSE_total={best_rmse_II:.5f}  -> {CKPT_PHASE_II_BEST.name}')

    _d = _check_dag_drift(f'Phase-II-ep{ep}')
    assert _d < CRITERION_DAG_DRIFT, f'A_dag a bouge ! drift={_d:.2e}'

print(f'\n[Cell 8] Phase II terminee. best RMSE_total={best_rmse_II:.5f}')

# Charger le best pour la suite (Phase III)
if CKPT_PHASE_II_BEST.exists():
    _ck_best = torch.load(CKPT_PHASE_II_BEST, map_location=DEVICE, weights_only=False)
    dual_path.gate.load_state_dict(_ck_best['gate_state_dict'])
    print(f'  -> Gate chargee depuis BEST (RMSE_total={_ck_best["best_rmse_total"]:.5f})')


In [ ]:
# === Cell 9 : Phase III  Joint fine-tuning (A_dag gele, avec resume) ===
print('=' * 60)
print('PHASE III : Joint (backbone + DualPath, A_dag gele)')
print(f'  {PHASE_III_EPOCHS} epochs')
print(f'  LR backbone={PHASE_III_LR_BACKBONE}  LR dualpath={PHASE_III_LR_DUALPATH}')
print(f'  lambda_causal : {LAMBDA_CAUSAL_START} -> {LAMBDA_CAUSAL_END}')
print('=' * 60)

# --- Setup gels : backbone unfreeze SAUF A_dag ---
_unfreeze_path_a_keep_dag()
for p in dual_path.parameters():
    p.requires_grad_(True)

if _A_dag_ref is not None:
    assert not _rcn_core.A_dag.requires_grad, 'A_dag doit etre gele !'
    print(f'[Cell 9] A_dag.requires_grad = {_rcn_core.A_dag.requires_grad}  OK')

# Optimizer multi-groupe
optimizer_III = Adam([
    {
        'params': list(encoder.parameters()) + list(rcn_runner.parameters()) + list(regression_head.parameters()),
        'lr': PHASE_III_LR_BACKBONE,
    },
    {
        'params': list(dual_path.parameters()),
        'lr': PHASE_III_LR_DUALPATH,
    },
])

# --- Resume detection ---
_start_ep_III     = 1
best_rmse_total   = float('inf')
phase_III_metrics = []

if RESUME and CKPT_PHASE_III_LAST.exists():
    print(f'[Cell 9] RESUME : checkpoint trouve {CKPT_PHASE_III_LAST}')
    _ck_III = torch.load(CKPT_PHASE_III_LAST, map_location=DEVICE, weights_only=False)
    encoder.load_state_dict(_ck_III['encoder_state_dict'])
    rcn_cell.load_state_dict(_ck_III['rcn_cell_state_dict'])
    regression_head.load_state_dict(_ck_III['regression_head_state_dict'])
    dual_path.load_state_dict(_ck_III['dual_path_state_dict'])
    optimizer_III.load_state_dict(_ck_III['optimizer_III_state_dict'])
    _start_ep_III     = int(_ck_III.get('epoch', 0)) + 1
    best_rmse_total   = float(_ck_III.get('best_rmse_total', float('inf')))
    phase_III_metrics = list(_ck_III.get('phase_III_metrics', []))
    # Re-geler A_dag apres load
    if _A_dag_ref is not None:
        _rcn_core.A_dag.requires_grad_(False)
    print(f'  Reprise a epoch {_start_ep_III}/{PHASE_III_EPOCHS}  '
          f'best_rmse_total={best_rmse_total:.5f}')

    if _start_ep_III > PHASE_III_EPOCHS:
        print(f'[Cell 9] Phase III deja terminee.')

OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Training loop ---
for ep in range(_start_ep_III, PHASE_III_EPOCHS + 1):
    frac = (ep - 1) / max(PHASE_III_EPOCHS - 1, 1)
    lambda_c = LAMBDA_CAUSAL_START * (1 - frac) + LAMBDA_CAUSAL_END * frac

    # Re-forcer A_dag.requires_grad=False (securite avant chaque epoch)
    if _A_dag_ref is not None:
        _rcn_core.A_dag.requires_grad_(False)

    print(f'\n--- Phase III epoch {ep}/{PHASE_III_EPOCHS}  lambda_c={lambda_c:.3f} ---')

    m_III = train_epoch_dualpath_phase3(
        dual_path=dual_path,
        optimizer=optimizer_III,
        data_loader=iterate_batches(train_dataloader, builder, DEVICE),
        device=DEVICE,
        encoder=encoder,
        rcn_runner=rcn_runner,
        regression_head=regression_head,
        rcn_cell=rcn_cell,
        builder=builder,
        lambda_causal=lambda_c,
        lambda_div=LAMBDA_DIV_III,
        gradient_clipping=GRADIENT_CLIPPING,
        log_interval=30,
        use_amp=USE_AMP,
        verbose=True,
    )
    print(f'  loss_main={m_III["loss_main"]:.5f}  loss_causal={m_III["loss_causal"]:.5f}  loss_div={m_III["loss_div"]:.5f}')

    # Re-forcer A_dag apres backward (securite doublee)
    if _A_dag_ref is not None:
        _rcn_core.A_dag.requires_grad_(False)

    # Verifier A_dag drift
    _drift = _check_dag_drift(f'Phase-III-ep{ep}')
    assert _drift < CRITERION_DAG_DRIFT, f'A_dag a bouge ! drift={_drift:.2e}'

    # Probe val
    _p = _probe_dualpath(f'Phase-III-ep{ep}', n_max=60)
    phase_III_metrics.append({**m_III, **_p, 'lambda_c': lambda_c, 'dag_drift': _drift})

    # --- Persistance : last + best (Phase III) ---
    _ck_out = {
        'schema_version':               3,
        'phase':                        'III',
        'epoch':                        ep,
        'total_epochs':                 PHASE_I_EPOCHS + PHASE_II_EPOCHS + ep,
        'encoder_state_dict':           encoder.state_dict(),
        'rcn_cell_state_dict':          rcn_cell.state_dict(),
        'regression_head_state_dict':   regression_head.state_dict(),
        'dual_path_state_dict':         dual_path.state_dict(),
        'optimizer_III_state_dict':     optimizer_III.state_dict(),
        'dual_path_config': {
            'in_channels':   C_LR,
            'base_ch':       PATH_B_BASE_CH,
            'hr_h':          H_HR,
            'hr_w':          W_HR,
            'gate_max_mean': GATE_MAX_MEAN,
        },
        'rmse_A_baseline':   RMSE_A_BASELINE,
        'phase_I_losses':    phase_I_losses,
        'phase_II_metrics':  phase_II_metrics,
        'phase_III_metrics': phase_III_metrics,
        'best_rmse_total':   best_rmse_total,
    }
    torch.save(_ck_out, CKPT_PHASE_III_LAST)
    # Aussi un checkpoint par epoch pour archive
    _ck_archive = OUT_DIR / f'dualpath_phase3_ep{ep:02d}.pth'
    torch.save(_ck_out, _ck_archive)
    print(f'  Saved last -> {CKPT_PHASE_III_LAST.name}')

    if _p['rmse_total'] < best_rmse_total:
        best_rmse_total = _p['rmse_total']
        _ck_out['best_rmse_total'] = best_rmse_total
        torch.save(_ck_out, CKPT_PHASE_III_BEST)
        torch.save(_ck_out, CKPT_DUAL_OUT)
        print(f'  * NEW best RMSE_total={best_rmse_total:.5f}  -> {CKPT_PHASE_III_BEST.name}  (+ alias)')

print(f'\n[Cell 9] Phase III terminee. best_rmse_total={best_rmse_total:.5f}')

# Charger le best Phase III pour Cells 10-12 (probe final, sigma, verdict)
if CKPT_PHASE_III_BEST.exists():
    _ck_best = torch.load(CKPT_PHASE_III_BEST, map_location=DEVICE, weights_only=False)
    encoder.load_state_dict(_ck_best['encoder_state_dict'])
    rcn_cell.load_state_dict(_ck_best['rcn_cell_state_dict'])
    regression_head.load_state_dict(_ck_best['regression_head_state_dict'])
    dual_path.load_state_dict(_ck_best['dual_path_state_dict'])
    if _A_dag_ref is not None:
        _rcn_core.A_dag.requires_grad_(False)
    print(f'  -> Stack charge depuis BEST (RMSE_total={_ck_best["best_rmse_total"]:.5f})')


In [ ]:
# === Cell 10 : Probe final complet sur tout val_dataset ===
print('=== Probe FINAL sur tout le val_dataset ===')

probe_final = _probe_dualpath('FINAL')

# Verifier causal_frac
_cf_ok = probe_final['causal_frac'] >= CRITERION_CAUSAL_FRAC
print(f'\n  causal_frac={probe_final["causal_frac"]:.3f}  '
      f'(seuil>={CRITERION_CAUSAL_FRAC}) : {"OK" if _cf_ok else "WARN"}')

# Tableau comparatif
print('\nTableau comparatif :')
print(f'{"Metrique":<22} {"Baseline (mu_A)":>18} {"Phase I (mu_B)":>18} {"Final (mu_total)":>18}')
print('-' * 78)
print(f'{"RMSE":<22} {RMSE_A_BASELINE:>18.5f} {probe_after_I["rmse_B"]:>18.5f} {probe_final["rmse_total"]:>18.5f}')
print(f'{"std":<22} {probe_final["std_A"]:>18.4f} {probe_final["std_B"]:>18.4f} {probe_final["std_total"]:>18.4f}')
print(f'{"gate_mean":<22} {"-":>18} {"-":>18} {probe_final["gate_mean"]:>18.4f}')
print(f'{"causal_frac":<22} {1.0:>18.3f} {probe_after_I["causal_frac"]:>18.3f} {probe_final["causal_frac"]:>18.3f}')
print('-' * 78)

# Verifications
_dag_drift_final = _check_dag_drift('final')
print(f'\n  A_dag drift final : {_dag_drift_final:.1e}')

In [ ]:
# === Cell 11 : Recalibration sigma_data Stage 2 ===
# Stage 2 EDM Karras est conditionne sur mu_HR.
# sigma_data = std(HR_true - mu_total)

print('=== Recalibration sigma_data pour Stage 2 ===')

@torch.no_grad()
def _calibrate_sigma_dualpath():
    encoder.eval(); rcn_cell.eval(); regression_head.eval(); dual_path.eval()
    n, mean_acc, m2 = 0, 0.0, 0.0
    for bi, batch_list in enumerate(iterate_batches(val_dataloader, builder, DEVICE)):
        for micro in batch_list:
            tgt = micro['residual'][-1].to(DEVICE)
            if tgt.dim() == 3: tgt = tgt.unsqueeze(0)
            mu_A, mu_B, mu_total, gate = predict_mu_hr_dualpath(
                micro,
                encoder=encoder, rcn_runner=rcn_runner,
                regression_head=regression_head, dual_path=dual_path,
                builder=builder, device=DEVICE,
                target_shape=tgt.shape[-2:],
            )
            delta = tgt - mu_total
            vals = delta[torch.isfinite(delta)].float().flatten()
            if vals.numel() == 0:
                continue
            stride = max(1, vals.numel() // 512)
            for x in vals[::stride].cpu().tolist():
                n += 1
                diff = x - mean_acc
                mean_acc += diff / n
                m2 += diff * (x - mean_acc)
    sigma_new = (m2 / max(1, n - 1)) ** 0.5
    return sigma_new, mean_acc, n

sigma_new, mean_res, n_pix = _calibrate_sigma_dualpath()

# Valeur de reference (Phase 5 / causal seul)
sigma_old = 0.18
try:
    _sigma_sre_p = OUT_DIR / 'sigma_data_sre.json'
    if _sigma_sre_p.exists():
        sigma_old = json.loads(_sigma_sre_p.read_text()).get('sigma_data_new', 0.18)
        print(f'  sigma_old (Phase 5 SRE) = {sigma_old:.5f}')
except Exception:
    pass

print(f'  sigma_data ancien (Phase 5)   : {sigma_old:.5f}')
print(f'  sigma_data nouveau (DualPath)  : {sigma_new:.5f}')
print(f'  mean residuel                  : {mean_res:.5f}')
print(f'  n_pixels_eval                  : {n_pix:,}')

_ratio = sigma_new / sigma_old if sigma_old > 0 else float('inf')
if _ratio > 1.30 or _ratio < 0.70:
    print(f'  WARNING : Changement > 30% (ratio={_ratio:.3f}) !')
    print('    -> Stage 2 doit etre reentraine from scratch.')
elif _ratio > 1.10 or _ratio < 0.90:
    print(f'  WARN : Changement > 10% (ratio={_ratio:.3f}) : mettre a jour sigma_data Stage 2.')
else:
    print(f'  OK : Changement < 10% (ratio={_ratio:.3f}) — Stage 2 reste compatible.')

sigma_result = {
    'sigma_data_old': sigma_old,
    'sigma_data_new': float(sigma_new),
    'mean_residual':  float(mean_res),
    'n_pixels':       int(n_pix),
    'ratio':          float(_ratio),
    'source':         'phase6_dualpath',
}
SIGMA_JSON.write_text(json.dumps(sigma_result, indent=2))
print(f'\nSauvegarde -> {SIGMA_JSON}')

In [ ]:
# === Cell 12 : Verdict GO / NO-GO ===
print('=' * 70)
print('PHASE 6 DUAL-PATH — VERDICT FINAL')
print('=' * 70)

_dag_drift_final = _check_dag_drift('verdict')

# Criteres
C1 = probe_final['rmse_total'] < RMSE_A_BASELINE * CRITERION_RMSE_IMPROVE
C2 = probe_final['rmse_total'] < CRITERION_RMSE_ABS
C3 = probe_final['causal_frac'] >= CRITERION_CAUSAL_FRAC
C4 = _dag_drift_final < CRITERION_DAG_DRIFT
C5 = probe_final['std_total'] > probe_final['std_A']

# Tableau criteres
_fmt = '{:<50} {:>6}  {:<20}'
print()
print(_fmt.format('Critere', 'Valeur', 'Resultat'))
print('-' * 78)
print(_fmt.format(
    f'[1] RMSE(mu_total) < RMSE(mu_A) * {CRITERION_RMSE_IMPROVE} (amelioration >=5%)',
    f'{probe_final["rmse_total"]:.5f}',
    f'PASS  (seuil {RMSE_A_BASELINE * CRITERION_RMSE_IMPROVE:.5f})' if C1
    else f'FAIL  (seuil {RMSE_A_BASELINE * CRITERION_RMSE_IMPROVE:.5f})'
))
print(_fmt.format(
    f'[2] RMSE(mu_total) < {CRITERION_RMSE_ABS} (seuil critique)',
    f'{probe_final["rmse_total"]:.5f}',
    'PASS' if C2 else 'FAIL'
))
print(_fmt.format(
    f'[3] causal_frac >= {CRITERION_CAUSAL_FRAC} (causalite preservee)',
    f'{probe_final["causal_frac"]:.3f}',
    'PASS' if C3 else 'FAIL'
))
print(_fmt.format(
    f'[4] A_dag drift < {CRITERION_DAG_DRIFT:.0e} (graphe causal intact)',
    f'{_dag_drift_final:.1e}',
    'PASS' if C4 else 'FAIL'
))
print(_fmt.format(
    '[5] std(mu_total) > std(mu_A) (meilleure couverture)',
    f'{probe_final["std_total"]:.4f}',
    f'PASS  (mu_A={probe_final["std_A"]:.4f})' if C5
    else f'FAIL  (mu_A={probe_final["std_A"]:.4f})'
))
print('-' * 78)

_all_pass = C1 and C2 and C3 and C4 and C5
_core_pass = C1 and C3 and C4

if _all_pass:
    print('\nVERDICT : GO')
    print('  -> Reentraine Stage 2 avec :')
    print(f'     sigma_data = {sigma_result["sigma_data_new"]:.5f}')
    print(f'     checkpoint = {CKPT_DUAL_OUT}')
    print('  -> Prochaine etape : phase7_stage2_retrain.ipynb')
elif _core_pass:
    print('\nVERDICT : GO PARTIEL (criteres principaux OK, voir details)')
    if not C2:
        print(f'  [2] RMSE_total={probe_final["rmse_total"]:.5f} >= {CRITERION_RMSE_ABS} — verifier la qualite absolue')
    if not C5:
        print(f'  [5] std(mu_total)={probe_final["std_total"]:.4f} <= std(mu_A)={probe_final["std_A"]:.4f}')
        print('      -> Augmenter PHASE_III_EPOCHS ou PHASE_III_LR_DUALPATH')
    print(f'     sigma_data = {sigma_result["sigma_data_new"]:.5f}')
    print(f'     checkpoint = {CKPT_DUAL_OUT}')
else:
    print('\nVERDICT : NO-GO  — Diagnostics :')
    if not C1:
        print(f'  [1] RMSE(mu_total)={probe_final["rmse_total"]:.5f} vs seuil {RMSE_A_BASELINE * CRITERION_RMSE_IMPROVE:.5f}')
        print('      -> Phase I insuffisante : essayer PHASE_I_EPOCHS=10')
        print('      -> Ou augmenter PHASE_III_LR_DUALPATH=5e-4')
    if not C3:
        print(f'  [3] causal_frac={probe_final["causal_frac"]:.3f} < {CRITERION_CAUSAL_FRAC}')
        print('      -> Augmenter LAMBDA_CAUSAL_START ou LAMBDA_DIV')
    if not C4:
        print(f'  [4] A_dag a derive ({_dag_drift_final:.2e}) !')
        print('      -> BUG CRITIQUE : verifier _rcn_core.A_dag.requires_grad_(False)')

# Sauvegarder resume JSON
_summary = {
    'verdict': 'GO' if _all_pass else ('GO_PARTIAL' if _core_pass else 'NO_GO'),
    'rmse_A_baseline': RMSE_A_BASELINE,
    'probe_after_phase_I': {k: float(v) for k, v in probe_after_I.items()},
    'probe_final': {k: float(v) for k, v in probe_final.items()},
    'sigma_data': sigma_result,
    'dag_drift_final': float(_dag_drift_final),
    'criteria': {'C1': C1, 'C2': C2, 'C3': C3, 'C4': C4, 'C5': C5},
    'hparams': {
        'phase_I_epochs': PHASE_I_EPOCHS, 'phase_I_lr': PHASE_I_LR,
        'phase_II_epochs': PHASE_II_EPOCHS, 'phase_II_lr': PHASE_II_LR,
        'phase_III_epochs': PHASE_III_EPOCHS,
        'lr_backbone': PHASE_III_LR_BACKBONE, 'lr_dualpath': PHASE_III_LR_DUALPATH,
        'lambda_causal_start': LAMBDA_CAUSAL_START, 'lambda_causal_end': LAMBDA_CAUSAL_END,
        'lambda_div': LAMBDA_DIV_III, 'gate_max_mean': GATE_MAX_MEAN,
    },
    'checkpoint': str(CKPT_DUAL_OUT),
}
(OUT_DIR / 'phase6_summary.json').write_text(json.dumps(_summary, indent=2))
print(f'\nResume sauvegarde -> {OUT_DIR}/phase6_summary.json')

if _all_pass or _core_pass:
    print(f'\n-> Reentraine Stage 2 avec sigma_data={sigma_result["sigma_data_new"]:.5f}')
print('=' * 70)